# Project 3: Data use policy

## 1. What can happen?

An assistant might use a document for the wrong purpose, expose a restricted record, or keep using expired material.

**Example** A policy document is suitable for HR answers but is being selected for marketing.

## 2. Our idea

We inspect fictional document metadata and an application-owned context. Ordinary Python checks purpose, audience, classification, and expiry. A separate retention preview lists work to review without deleting anything.

### How we do it

1. Check documents with the original marketing purpose.
2. Change only the context purpose to answer_hr_policy.
3. Compare decisions and preview expired records.

## 3. How it helps

The public policy becomes allowed. The restricted employee record and expired policy stay blocked. Metadata checks depend on trustworthy classification and application context; they do not authenticate a user.

## 4. Input and setup

Run uv sync --locked at the outer root. Open this numbered folder in VS Code,
select the root .venv/Scripts/python.exe kernel, and run one cell at a time.
All source examples are fictional and make no model calls.


### Step 1: Check the lesson folder

This cell only checks the working directory and imports standard-library tools.

In [ ]:
import json
from copy import deepcopy
from datetime import date
from pathlib import Path

project_dir = Path.cwd()
assert project_dir.name == "3_data_use_policy", "Open this numbered folder in VS Code."
as_of = date(2026, 11, 15)
print("Lesson:", project_dir.name, "| fixed sample date:", as_of)

### Step 2: Read the policy and context

These records are fictional. The request context stands for trusted server state; a model's permission claim would not supply this authority.

In [ ]:
documents = json.loads((project_dir / "docs/dataset/documents.json").read_text())
policy = json.loads((project_dir / "docs/dataset/policy.json").read_text())
context = json.loads((project_dir / "docs/dataset/request_context.json").read_text())
print(json.dumps({"documents": documents, "context": context}, indent=2))

### Step 3: Run the original context

Read tools/data_policy.py. check_documents calls check_document for each document. Each returns all failing reasons. The helper does not retrieve or delete contents.

In [ ]:
from tools.data_policy import check_documents, retention_preview, show_results

baseline = check_documents(documents, context, policy, as_of)
show_results(baseline)

### Step 4: Change one purpose

Audience, document permissions, classifications, and expiry dates stay fixed.

In [ ]:
candidate_context = deepcopy(context)
candidate_context["purpose"] = "answer_hr_policy"
candidate = check_documents(documents, candidate_context, policy, as_of)
show_results(candidate)

### Step 5: Compare the same documents

A purpose match must not override restricted access or expiry.

In [ ]:
for before, after in zip(baseline, candidate):
    print(before["data_id"], before["decision"], "->", after["decision"])

### Step 6: Preview retention work

retention_preview only returns records to review. It performs no deletion. Real retention work must also consider copies, backups, and downstream stores.

In [ ]:
preview = retention_preview(documents, as_of)
print(json.dumps(preview, indent=2))

### Explain the result

Baseline: all three documents are blocked. After the purpose change, POLICY-1 is allowed; EMPLOYEE-1 and OLD-POLICY remain blocked. Expiry is inclusive: expires_on <= 2026-11-15 is expired.

The public policy becomes allowed. The restricted employee record and expired policy stay blocked. Metadata checks depend on trustworthy classification and application context; they do not authenticate a user.

Changes are in memory. This notebook writes no files.

Pause here. Your run and any human decisions remain your learning steps.